# Gemma 4 on Vertex AI — deploy, then personalize (workshop handout)

A hands-on notebook: deploy an open **Gemma 4** model as your own private endpoint,
then personalize it with **RAG** and **fine-tuning**.

**How to use this notebook**
1. Fill in the **Setup** section with your own GCP project + endpoint (there are two
   `TODO` placeholders).
2. Run top to bottom. The inference cells call *your* endpoint.
3. The fine-tuning section is real training code — run it in a **GPU** runtime with
   your own dataset (see notes there).

> Requirements: a Google Cloud project with Vertex AI enabled, and permission to
> deploy from Model Garden. The fine-tuning section additionally needs a GPU and a
> Hugging Face account that has accepted the Gemma license.

> **💸 Cost — please read first.** Running this end-to-end costs on the order of **a
> few tens of USD** — *but only if you shut the endpoint down when you finish.* A
> deployed endpoint **bills continuously, per hour, whether or not you use it.** Left
> running it can cost **hundreds to thousands of USD per month** (especially GPU
> machines). **Always run the Clean up step (Section 10) when you're done.**

---
> **Disclaimer:** This code is provided "as-is" as a demonstration only to illustrate a
> potential solution. The code does not constitute a Google product or service of any
> kind, and Google offers no support, warranties, or liability of any kind with its
> regard. Whoever chooses to use this code accepts all responsibility related to it,
> including for its implementation, use, and ongoing maintenance. For the avoidance of
> doubt, this code is not eligible for the Google Open Source Software Vulnerability
> Rewards Program.
---

## 1 · What Gemma is, and picking a size

**Gemma** is Google's family of **open-weight** models — same research lineage as
Gemini, but the weights are downloadable and run on your own infrastructure.
**Gemma 4** is the current generation and is **multimodal** (text + images).

| Size | Character | Typical hardware |
|---|---|---|
| **E2B / E4B** | Efficient, "elastic" (MatFormer) — edge & low cost | CPU or a single small GPU |
| **12B** | Balanced quality | One large GPU (e.g. H100) |
| **31B** | Highest quality | One or more large GPUs |

The number is **billions of parameters**; more parameters ≈ higher quality but more
cost, latency, and hardware. The "**E**" stands for **effective parameters**: E4B has
~8B total parameters but runs with only ~4.5B active at inference (via **Per-Layer
Embeddings**), giving bigger-model quality at a ~4B memory footprint.
**Rule of thumb: pick the smallest size that clears your quality bar.**

This notebook uses an **E4B** endpoint (it can even run on CPU). Everything here —
RAG and fine-tuning — works identically on any size and scales straight up.

### Under the hood, in 60 seconds

- **Transformer** — the architecture under Gemma. It reads text as **tokens** and
  predicts the next one using **attention** (learning which earlier tokens matter).
  Stacked and trained on a large corpus, it learns language and facts; generation is
  "predict the next token," repeatedly.
- **MatFormer** — smaller, fully-working models are **nested inside the bigger one,
  like Russian dolls**, so one checkpoint can be served at several sizes without
  retraining. (The "E" itself stands for *effective* parameters; see above.)

## 2 · Setup — deploy your own Gemma 4 endpoint

**Deploy the model (one time):**
1. Go to **Vertex AI → Model Garden**, open **Gemma 4**, click **Deploy**.
2. Pick a size/machine your project has quota for (E4B can deploy on CPU; larger
   sizes need a GPU). Accept the license and deploy. It takes ~20 min.
3. When it's live, open **Vertex AI → Online prediction → Endpoints**, click your
   endpoint, and copy its **Endpoint ID**.

Then fill in the two placeholders below.

In [ ]:
# TODO (1): your Google Cloud project id
PROJECT = "YOUR_PROJECT_ID"
# TODO (2): the endpoint id you copied from Vertex AI > Online prediction
ENDPOINT_ID = "YOUR_ENDPOINT_ID"

REGION = "us-central1"   # change if you deployed elsewhere
assert PROJECT != "YOUR_PROJECT_ID" and ENDPOINT_ID != "YOUR_ENDPOINT_ID", \
    "Fill in PROJECT and ENDPOINT_ID above first."
print("Project:", PROJECT, "| Endpoint:", ENDPOINT_ID)

## 3 · Connect to your model

Authenticates with the notebook's own Google credentials (no API keys). The
endpoint's dedicated DNS is **auto-discovered**, so this keeps working even if you
redeploy (the DNS changes each time).

In [ ]:
import google.auth, google.auth.transport.requests, requests
from google.cloud import aiplatform

aiplatform.init(project=PROJECT, location=REGION)
_ep = aiplatform.Endpoint(ENDPOINT_ID)
# Dedicated endpoints have their own DNS; shared endpoints use the regional host.
DEDICATED_DNS = getattr(_ep.gca_resource, "dedicated_endpoint_dns", "") \
    or f"{REGION}-aiplatform.googleapis.com"
print("Serving host:", DEDICATED_DNS)

_creds, _ = google.auth.default()
def _token():
    _creds.refresh(google.auth.transport.requests.Request())
    return _creds.token

EP  = f"projects/{PROJECT}/locations/{REGION}/endpoints/{ENDPOINT_ID}"
URL = f"https://{DEDICATED_DNS}/v1beta1/{EP}:predict"

def chat(messages, max_tokens=220, temperature=0.2):
    body = {"instances": [{
        "@requestFormat": "chatCompletions",
        "messages": messages, "max_tokens": max_tokens, "temperature": temperature,
    }]}
    r = requests.post(URL, headers={"Authorization": f"Bearer {_token()}"},
                      json=body, timeout=300)
    r.raise_for_status()
    pred = r.json()["predictions"]
    pred = pred if isinstance(pred, dict) else pred[0]
    return pred["choices"][0]["message"]["content"].strip()

def u(text):
    return [{"role": "user", "content": [{"type": "text", "text": text}]}]

print(chat(u("In one sentence, what is Gemma?"), max_tokens=60))

## 4 · Self-hosted Gemma vs. the Gemini API

| | **Gemini API** (managed) | **Gemma, self-deployed** |
|---|---|---|
| Setup | One API call, no infra | Deploy an endpoint |
| Data boundary | Google's managed service | Your project / VPC |
| Customization | Prompting, context caching | Prompting **+ fine-tune the weights** |
| Cost shape | Per token | Per hour the endpoint runs |
| Runs on-prem / edge / offline | No | **Yes** (open weights) |
| Best when | Frontier quality, fastest to ship | Control, privacy, customization |

Reach for self-hosted Gemma when you need **privacy/control**, **customization**
(change the weights, not just the prompt), or **reach** (run where an API can't).

## 5 · Personalization #1 — RAG (give it your facts)

**RAG** doesn't change the model — it changes the **prompt**. At question time you
retrieve the most relevant passages from your own documents and paste them into the
context, so the model answers from *your* facts and can **cite** them.

Pattern: **corpus → index (embeddings + vector DB) → retrieve → augment prompt →
generate**. Below, the corpus is a few strings and retrieval is simple word-overlap
so it runs with no extra services; in production, swap in **EmbeddingGemma** + a
vector database (e.g. Vertex AI Vector Search).

In [ ]:
import re

# (1) CORPUS — replace with your own documents (from a wiki, Drive, a DB, ...).
DOCS = [
    "ExampleCo PTO policy: Full-time employees accrue 1.5 vacation days per month, "
    "up to a maximum balance of 30 days. Unused days above 30 are forfeited each Dec 31.",
    "ExampleCo expense policy: Meals during travel are reimbursed up to $75/day domestic "
    "and $110/day international. Receipts required for any single expense over $25.",
    "The X2 sensor ships with firmware 4.2. Operating temperature range is -10C to 55C.",
]

# (3) RETRIEVE — production: EmbeddingGemma + a vector DB. Word-overlap here.
_STOP = set("a an the of to and or for in on at is are be how what can i my you your with up".split())
def retrieve(query, k=2):
    qt = {w for w in re.findall(r"[a-z0-9]+", query.lower()) if w not in _STOP}
    scored = [(len(qt & set(re.findall(r"[a-z0-9]+", d.lower()))), d) for d in DOCS]
    scored.sort(reverse=True)
    return [d for s, d in scored[:k]]

print("Corpus:", len(DOCS), "documents")

In [ ]:
Q = "How many vacation days can I carry, and what happens to extras at year end?"

without_rag = chat([
    {"role": "system", "content": [{"type": "text", "text": "You are ExampleCo's internal assistant."}]},
    {"role": "user",   "content": [{"type": "text", "text": Q}]},
], max_tokens=120)

# (4) AUGMENT + GENERATE: retrieve, paste as context, instruct to cite.
ctx = "\n\n".join(f"[Doc {i+1}] {d}" for i, d in enumerate(retrieve(Q)))
with_rag = chat([
    {"role": "system", "content": [{"type": "text", "text":
        "You are ExampleCo's internal assistant. Answer ONLY from the provided context. "
        "If it isn't in the context, say you don't have that information. Cite the doc number(s)."}]},
    {"role": "user",   "content": [{"type": "text", "text": f"Context:\n{ctx}\n\nQuestion: {Q}"}]},
], max_tokens=160)

print("WITHOUT RAG:\n", without_rag)
print("\n" + "="*70 + "\nWITH RAG (grounded + cited):\n", with_rag)

Without retrieval the model guesses or invents; with retrieval it answers from your
document **and cites it**, and returns "I don't have that information" when the
answer isn't present. Update a document → the answer updates, with no retraining.
**Use RAG for facts that change** and anywhere you need citations.

## 6 · Personalization #2 — Fine-tuning with LoRA (real training code)

RAG changes what the model *knows*; **fine-tuning changes how it behaves** (tone,
format, a skill) by adjusting the weights. **LoRA** freezes the base weights and
trains a small **adapter** (a few MB) — cheap and fast.

**What you need:**
- A **GPU runtime** (this won't run on a CPU-only or endpoint-only runtime).
- A **Hugging Face account** that has accepted the Gemma license, and a token.
- **Your dataset**: a few hundred+ curated `input → ideal response` examples that
  demonstrate the behavior you want. Training *produces* the adapter — you don't
  start with one.

> The cells below are **real and runnable in a GPU environment**. Verify the exact
> model id on the live Gemma model card before running.

In [ ]:
# 1) Install training libraries (GPU runtime).
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

In [ ]:
# 2) Authenticate to Hugging Face (accept the Gemma license on the model card first).
import os
os.environ["HF_TOKEN"] = "hf_xxx"   # <- your token, read scope

# 3) Your curated dataset: replace with a few hundred+ real examples.
from datasets import Dataset
raw = [
    {"messages": [
        {"role": "user",      "content": "A customer says the X2 sensor is overheating."},
        {"role": "assistant", "content": "Support ticket draft — Likely cause: ambient above the "
                                         "55C max. Steps: 1) confirm ambient temp, 2) reduce load, "
                                         "3) if it persists, start an RMA."},
    ]},
    # ... add many more, in YOUR voice/format ...
]
dataset = Dataset.from_list(raw)
print("training examples:", len(dataset))

In [ ]:
# 4) Load the base model + tokenizer.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/gemma-4-e4b-it"   # verify the exact id on the model card
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

In [ ]:
# 5) Configure LoRA and train. Produces a small adapter (a few MB).
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora,
    args=SFTConfig(
        output_dir="gemma4-myvoice",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        max_seq_length=1024,
    ),
)
trainer.train()
trainer.save_model("gemma4-myvoice")   # <- your adapter is saved here
print("Adapter saved to ./gemma4-myvoice")

In [ ]:
# 6) Use it: load the base model + your adapter, then generate.
from peft import PeftModel
tuned = PeftModel.from_pretrained(model, "gemma4-myvoice")

msgs = [{"role": "user", "content": "A customer says the X2 sensor is overheating."}]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(tuned.device)
out = tuned.generate(inputs, max_new_tokens=200)
print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))

**Evaluate before vs. after.** Keep a small held-out set of prompts and compare the
base model's answers to the fine-tuned model's — that's how you confirm the behavior
actually changed the way you wanted. Fine-tune for **voice, format, and skills**; use
**RAG** for facts that change.

## 7 · RAG vs. fine-tuning

| | **RAG** | **Fine-tuning (LoRA)** |
|---|---|---|
| Changes | What it **knows** | How it **behaves** |
| Update cost | Cheap, instant | Retrain the adapter |
| Freshness | Always current | Goes stale |
| Citations | Yes | No |
| Best for | Facts that change | Voice, format, skills |

The strongest pattern is **both**: fine-tune the voice, RAG the facts.

## 8 · A few tips

**Tip 1 — Structured (JSON) output: treat the model as an API.**

In [ ]:
import json
js = chat(u("Extract strict JSON with keys name, temp_min_c, temp_max_c "
            "from: 'The X2 sensor operates from -10C to 55C.' JSON only."),
          max_tokens=120, temperature=0.0)
print(js)
print("parsed ->", json.loads(re.search(r"\{.*\}", js, re.S).group(0)))

**Tip 2 — Long context: sometimes you don't need RAG.** The E-series has a **128K**
context window; **12B and 31B double it to 256K**. For a small, bounded corpus (one
handbook, a single API spec), paste the whole thing into the context and pair it with
**context caching**. RAG still wins for large or fast-changing corpora.

**Tip 3 — MatFormer elasticity: one model, several sizes.** The E-series nests smaller
models inside the bigger one, so from a single E4B checkpoint you can serve a smaller,
faster slice or the full model — **without re-downloading or retraining** — selecting
the size at serving time. (Note: this elasticity is *within* the E-series; moving to
12B or 31B is a different, larger model you deploy separately.)

## 9 · Summary

- Deploy **Gemma 4** as your **own** Vertex AI endpoint — open weights, in your
  project's boundary.
- **RAG** gives it your **facts** (current, cited); **fine-tuning** changes its
  **behavior** (voice, format). The best systems use both.
- Pick the smallest size that clears your quality bar; everything here scales up
  unchanged.

## 10 · Clean up — **IMPORTANT: stop billing** 🧹

**This is the most important step in the notebook.** Your endpoint **bills for every
hour a model is deployed**, whether or not you're sending requests — a GPU endpoint
left running can cost **hundreds to thousands of USD per month**. When you're done,
**undeploy the model** (and optionally delete the endpoint).

**Option A — Console:** Vertex AI → Online prediction → Endpoints → your endpoint →
**Undeploy** the model.

**Option B — run the cell below.**

In [ ]:
# ⚠️ RUN THIS WHEN YOU'RE FINISHED — it stops the billing.
from google.cloud import aiplatform
aiplatform.init(project=PROJECT, location=REGION)
ep = aiplatform.Endpoint(ENDPOINT_ID)
ep.undeploy_all()            # removes the deployed model -> compute charges stop
# ep.delete(force=True)      # uncomment to also delete the endpoint entirely
print("Done — model undeployed. Compute billing stopped.")